Język: [English](lab.en.ipynb) | **Polski**

# Laboratorium 01. Regresja liniowa wielu zmiennych

Materiał uzupełnia [`README.pl.md`](README.pl.md). Mechanizm z wykładu 1,
obejmujący regresję liniową, gradient i pętlę treningową, zostaje rozszerzony
na wiele cech oraz przypadek, w którym model liniowy nie wystarcza.




In [ ]:
import matplotlib.pyplot as plt
import torch

torch.manual_seed(7)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używane urządzenie: {device}")

## 1. Dane syntetyczne (gotowe)

Zbiór zawiera trzy cechy $x_1$, $x_2$, $x_3$ oraz wartość docelową $y$.
Dane są syntetyczne i zawierają szum. Ich generowanie nie jest częścią zadania.


In [ ]:
N = 400

x1 = torch.empty(N).uniform_(-3, 3)
x2 = torch.empty(N).uniform_(-3, 3)
x3 = torch.empty(N).uniform_(-3, 3)

w_true = torch.tensor([1.5, -2.0])
quad_coef = 0.8
b_true = 0.5
noise_std = 1.0

y = w_true[0] * x1 + w_true[1] * x2 + quad_coef * x3**2 + b_true + torch.randn(N) * noise_std

X = torch.stack([x1, x2, x3], dim=1)  # shape (N, 3)

X = X.to(device)
y = y.to(device)

print(f"X.shape={tuple(X.shape)}, y.shape={tuple(y.shape)}, device={X.device}")

## 2. Metryki i podział danych

Do zaimplementowania są trzy funkcje.

- `train_val_split(n_samples, val_fraction)` zwraca `(train_idx, val_idx)`.
  Indeksy `0..n_samples-1` są losowo permutowane przez `torch.randperm`,
  a część `val_fraction` trafia do zbioru walidacyjnego.
- `mse(y_pred, y_true)` oblicza błąd średniokwadratowy.
- `r2_score(y_pred, y_true)` oblicza $R^2 = 1 - SS_{\text{res}}/SS_{\text{tot}}$.

Te trzy funkcje są używane przez całą resztę notebooka.


In [ ]:
def train_val_split(n_samples, val_fraction=0.2):
    """Zwraca (train_idx, val_idx): losowy podział 0..n_samples-1."""
    # TODO: wylosuj permutację indeksów (torch.randperm) i podziel ją
    # na pierwsze `val_fraction * n_samples` (walidacja) i resztę (trening).
    raise NotImplementedError("TODO: train_val_split")


def mse(y_pred, y_true):
    """Błąd średniokwadratowy."""
    # TODO
    raise NotImplementedError("TODO: mse")


def r2_score(y_pred, y_true):
    """R^2 = 1 - SS_res / SS_tot."""
    # TODO
    raise NotImplementedError("TODO: r2_score")

In [ ]:
train_idx, val_idx = train_val_split(N, val_fraction=0.2)
X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]

print(f"N_train={len(train_idx)}, N_val={len(val_idx)}")

## 3. Baseline (gotowe)

Predykcja = średnia $y$ ze zbioru treningowego, niezależna od $x$.


In [ ]:
baseline_pred_train = torch.full_like(y_train, y_train.mean())
baseline_pred_val = torch.full_like(y_val, y_train.mean())

baseline_mse_val = mse(baseline_pred_val, y_val)
baseline_r2_val = r2_score(baseline_pred_val, y_val)

print(f"Baseline  MSE (val)={baseline_mse_val.item():.4f}  R2 (val)={baseline_r2_val.item():.4f}")

## 4. Model `forward`

Funkcja uogólnia `w * x + b` z wykładu na postać macierzową
$\hat{y} = Xw + b$, gdzie `X` ma kształt `(N, 3)`, `w` ma kształt `(3,)`,
a `b` jest skalarem (tensor kształtu `(1,)`).


In [ ]:
def forward(X, w, b):
    """Predykcja modelu liniowego: X @ w + b."""
    # TODO: iloczyn macierzowy X i w (operator @ albo torch.matmul), plus b.
    raise NotImplementedError("TODO: forward")

## 5. Rozwiązanie w postaci zamkniętej

Do `X` należy dopisać kolumnę jedynek, aby bias znalazł się w tym samym
układzie równań. Następnie można rozwiązać *normal equation*.

$$w^*_{\text{aug}} = (X_{\text{aug}}^\top X_{\text{aug}})^{-1} X_{\text{aug}}^\top y$$

Wartości `w`, czyli pierwsze trzy współczynniki, oraz `b`, czyli ostatni
współczynnik, powinny zostać zwrócone **osobno**. Dzięki temu pasują do
sygnatury `forward(X, w, b)` z sekcji 4. Do rozwiązania układu należy użyć
`torch.linalg.solve`, które jest stabilniejsze numerycznie od jawnego
odwracania macierzy.


In [ ]:
def closed_form_solution(X, y):
    """Zwraca (w, b) z normal equation na danych treningowych."""
    # TODO:
    # 1. dopisz kolumnę jedynek do X (torch.cat z torch.ones(N, 1))
    # 2. rozwiąż (X_aug^T X_aug) w_aug = X_aug^T y  za pomocą torch.linalg.solve
    # 3. rozdziel w_aug na w (pierwsze 3 elementy) i b (ostatni element)
    raise NotImplementedError("TODO: closed_form_solution")


w_closed, b_closed = closed_form_solution(X_train, y_train)
closed_form_val_loss = mse(forward(X_val, w_closed, b_closed), y_val)
print(f"Rozwiazanie zamkniete:  w={w_closed.tolist()}  b={b_closed.item():.4f}")
print(f"Strata (val) rozwiazania zamknietego: {closed_form_val_loss.item():.4f}")

## 6. Trening metodą spadku gradientu

Pętla treningowa z wykładu zostaje rozszerzona na wektorowe `w` o kształcie
`(3,)`. Kolejność pozostaje bez zmian. *Forward → strata → backward →
aktualizacja*. Rozwiązanie nie korzysta z `nn.Module` ani `optim`.


In [ ]:
learning_rate = 0.05
n_epochs = 300


def train_linear_model(X_train, y_train, learning_rate, n_epochs):
    """Zwraca (w, b, loss_history) wytrenowane spadkiem gradientu."""
    # TODO:
    # 1. zainicjuj w (shape (3,)) i b (shape (1,)) jako torch.zeros z
    #    requires_grad=True, na odpowiednim device
    # 2. w pętli po n_epochs: forward -> strata (mse) -> loss.backward()
    #    -> aktualizacja pod torch.no_grad() -> wyzerowanie gradientów
    # 3. zbieraj stratę każdej epoki do listy i zwróć (w, b, loss_history)
    raise NotImplementedError("TODO: train_linear_model")


w_gd, b_gd, loss_history = train_linear_model(X_train, y_train, learning_rate, n_epochs)
print(f"Gradient descent:  w={w_gd.tolist()}  b={b_gd.item():.4f}")

## 7. Porównanie strat (gotowe)


In [ ]:
gd_val_loss = mse(forward(X_val, w_gd, b_gd), y_val)

print(f"Baseline            strata (val) = {baseline_mse_val.item():.4f}")
print(f"Rozwiazanie zamkniete strata (val) = {closed_form_val_loss.item():.4f}")
print(f"Gradient descent     strata (val) = {gd_val_loss.item():.4f}")

plt.figure(figsize=(5, 4))
plt.plot(loss_history)
plt.xlabel("epoka")
plt.ylabel("strata (MSE, trening)")
plt.title("Krzywa straty — spadek gradientu")
plt.tight_layout()
plt.show()

## 8. Ewaluacja na zbiorze walidacyjnym (gotowe)


In [ ]:
y_pred_val = forward(X_val, w_gd, b_gd)

val_mse = mse(y_pred_val, y_val)
val_r2 = r2_score(y_pred_val, y_val)

print(f"Model (GD) na zbiorze walidacyjnym: MSE={val_mse.item():.4f}  R2={val_r2.item():.4f}")

## 9. Analiza błędów i interpretacja wykresu reszt


In [ ]:
residuals_val = (y_pred_val - y_val).detach().cpu()
features_val = X_val.detach().cpu()
feature_names = ["x1", "x2", "x3"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for i, ax in enumerate(axes):
    ax.scatter(features_val[:, i], residuals_val, s=15, alpha=0.6)
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_xlabel(feature_names[i])
    ax.set_title(f"reszty vs {feature_names[i]}")
axes[0].set_ylabel("reszta (ŷ - y)")
plt.tight_layout()
plt.show()

Odpowiedź w komórce Markdown powinna mieć 3–5 zdań. Która cecha ma
systematyczny, nielosowy wzorzec w resztach? Jak wygląda ten wzorzec?
Jakiej zależności model liniowy *strukturalnie* nie potrafi wyrazić?

_(Miejsce na odpowiedź.)_


## 10. Wnioski

Odpowiedzi należy zapisać w komórce Markdown i krótko uzasadnić.

1. Czy ten sam problem dałoby się rozwiązać lepiej, dodając nową *cechę*
   (np. kwadrat jednej z istniejących) bez zmiany samego modelu liniowego?
   Z czego wynika odpowiedź?

   _(Miejsce na odpowiedź.)_

2. Dlaczego porównanie z rozwiązaniem w postaci zamkniętej jest użytecznym
   testem poprawności pętli treningowej?

   _(Miejsce na odpowiedź.)_
